In [1]:
import pickle
import pandas as pd
import json
#NUMERICAL FEATURE BINS

#SElected Features
with open(r'D:\home loan credit risk\artifact\binning\prebin\selected_features.json', 'rb') as f:
    selected_features = json.load(f)
for i in range(len(selected_features)):
    selected_features[i] = selected_features[i].replace('_WOE','')
        
#iv_df = pd.read_csv(r'D:\home loan credit risk\artifact\binning\automatic\iv_df.csv')

In [2]:
with open(r'D:\home loan credit risk\artifact\binning\prebin\numerical_feature_bins.pkl', 'rb') as f:
   numerical_feature_bins = pickle.load(f)

In [3]:
import pandas as pd
X_train = pd.read_csv(r'D:\home loan credit risk\artifact\splits\X_train.csv')
for col in X_train.select_dtypes(include=["float64"]).columns:
    X_train[col] = pd.to_numeric(X_train[col], downcast="float")

In [4]:
y_train = pd.read_csv(r'D:\home loan credit risk\artifact\splits\y_train.csv')
y_train = y_train.iloc[:, 0]

# CATEGORICAL BINNING UPDATED

In [ ]:
import pandas as pd
import numpy as np

class CategoricalWOEBinner:
    """
    Categorical WOE Binning for credit risk
    - One bin per category
    - Group rare categories into 'RARE'
    - Fit WOE values using training data only
    - Apply same mapping to validation / test data
    """
    def __init__(self, id_col, rare_threshold=0.01):
        self.id_col = id_col
        self.rare_threshold = rare_threshold
        self.categorical_features = []
        self.cat_woe_maps = {}
        self.iv_df = None

    def fit(self, X_train_cat, y_train):
        """Fit WOE mappings using training data."""
        self.categorical_features = [c for c in X_train_cat.columns if c != self.id_col]
        cat_iv_records = []

        for col in self.categorical_features:
            train_series = X_train_cat[col].fillna("MISSING")
            freq = train_series.value_counts(normalize=True)
            rare_categories = freq[freq < self.rare_threshold].index.tolist()

            # Group rare categories
            train_binned = train_series.where(~train_series.isin(rare_categories), "RARE")
            df_tmp = pd.DataFrame({"cat": train_binned, "target": y_train})

            # Aggregate counts
            agg = df_tmp.groupby("cat").agg(
                Count=("target", "count"),
                Event=("target", "sum")
            ).reset_index()
            agg["Non-event"] = agg["Count"] - agg["Event"]

            total_count = agg["Count"].sum()
            total_good = agg["Non-event"].sum()
            total_bad = agg["Event"].sum()

            # Distributions
            agg["Count(%)"] = agg["Count"] / total_count
            agg["Event rate"] = agg["Event"] / agg["Count"]
            agg["dist_good"] = agg["Non-event"] / total_good
            agg["dist_bad"] = agg["Event"] / total_bad

            # WOE
            agg["WoE"] = np.where(
                (agg["dist_good"] == 0) | (agg["dist_bad"] == 0),
                0.0,
                np.log(agg["dist_good"] / agg["dist_bad"])
            )

            # IV per bin
            agg["IV"] = (agg["dist_good"] - agg["dist_bad"]) * agg["WoE"]
            iv_value = agg["IV"].sum()

            # Stats for transform
            stats = {}
            for _, row in agg.iterrows():
                stats[row["cat"]] = {
                    "Count": row["Count"],
                    "Event": row["Event"],
                    "Non-event": row["Non-event"],
                    "Count(%)": row["Count(%)"],
                    "Event rate": row["Event rate"],
                    "WoE": row["WoE"],
                    "IV": row["IV"]
                }

            # Ensure RARE exists
            if "RARE" not in stats:
                stats["RARE"] = {
                    "Count": 0,
                    "Event": 0,
                    "Non-event": 0,
                    "Count(%)": 0.0,
                    "Event rate": 0.0,
                    "WoE": 0.0,
                    "IV": 0.0
                }

            woe_map = dict(zip(agg["cat"], agg["WoE"]))

            if "RARE" not in woe_map:
                woe_map["RARE"] = 0.0

            self.cat_woe_maps[col] = {
                "woe_map": woe_map,
                "rare_categories": rare_categories,
                "stats": stats
            }

            cat_iv_records.append({"feature": col, "IV": iv_value})

        self.iv_df = pd.DataFrame(cat_iv_records).sort_values(by="IV", ascending=False).reset_index(drop=True)
        return self

    def transform(self, X_cat):
        """Apply learned WOE mappings to Data."""
        X_woe = pd.DataFrame(index=X_cat.index)
        X_woe[self.id_col] = X_cat[self.id_col]

        for col in self.categorical_features:
            if col not in self.cat_woe_maps:
                continue

            woe_map = self.cat_woe_maps[col]["woe_map"]
            rare_categories = self.cat_woe_maps[col]["rare_categories"]

            test_series = X_cat[col].fillna("MISSING")
            test_binned = test_series.where(~test_series.isin(rare_categories), "RARE")
            test_binned = test_binned.where(test_binned.isin(woe_map.keys()), "RARE")
            X_woe[col] = test_binned.map(woe_map)

        return X_woe

    def fit_transform(self, X_train_cat, y_train):
        self.fit(X_train_cat, y_train)
        return self.transform(X_train_cat)

    def get_categorical_woe_bins(self):
        """Return full summary table with Event/Non-event/Count(%) etc."""
        records = []
        for feature, info in self.cat_woe_maps.items():
            stats = info["stats"]
            rare_cats = info["rare_categories"]

            for bin_name, s in stats.items():
                records.append({
                    "feature": feature,
                    "Bin": bin_name,
                    "Count": s["Count"],
                    "Event": s["Event"],
                    "Non-event": s["Non-event"],
                    "Count(%)": s["Count(%)"],
                    "Event rate": s["Event rate"],
                    "WoE": s["WoE"],
                    "IV": s["IV"],
                    "is_rare_bin": bin_name == "RARE",
                    "rare_categories": ", ".join(rare_cats) if bin_name == "RARE" else None
                })
            

        return pd.DataFrame(records)

In [24]:
id_col = 'SK_ID_CURR'
cat_cols_ = X_train.select_dtypes(
       include=["object", "category", "bool"]
   ).columns.tolist()
cat_cols_ = list(dict.fromkeys([id_col] + cat_cols_))


In [25]:
# Select categorical columns from the actual training DataFrame
X_train_cat = X_train[cat_cols_]  # this is a DataFrame

# Initialize the WOE binner
cat_woe = CategoricalWOEBinner(id_col=id_col, rare_threshold=0.01)

# Fit-transform
X_train_woe = cat_woe.fit_transform(X_train_cat, y_train)

# To see the WOE table with count, event rate etc.
woe_summary = cat_woe.get_categorical_woe_bins()

In [29]:
woe_summary[woe_summary['feature']=='NAME_INCOME_TYPE']

,feature,Bin,Count,Event,Non-event,Count(%),Event rate,WoE,IV,is_rare_bin,rare_categories
17,NAME_INCOME_TYPE,Commercial associate,57273,4348,52925,0.232810,0.075917,0.066678,0.001007,False,None
18,NAME_INCOME_TYPE,Pensioner,44134,2365,41769,0.179401,0.053587,0.438894,0.028798,False,None
19,NAME_INCOME_TYPE,RARE,40,5,35,0.000163,0.125000,-0.486572,0.000047,True,"Unemployed, Student, Businessman, Maternity leave"
20,NAME_INCOME_TYPE,State servant,17518,1024,16494,0.071209,0.058454,0.346798,0.007412,False,None
21,NAME_INCOME_TYPE,Working,127043,12118,114925,0.516418,0.095385,-0.182894,0.018653,False,None


# OptBinning Temprory on index 

In [5]:
from src.logger import config_logger
import logging
logger = config_logger('module_05_feature_engineering.py')

In [6]:
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd


def manual_prebinning(
    df,
    feature,
    target,
    max_bins=20,
    min_bin_size=0.05,
    special_values=None,
    epsilon=1e-6
):

    if special_values is None:
        special_values = []

    y = target.loc[df.index]

    series = df[feature].copy()
    series = series.fillna(-99999)

    # Separate special & normal
    special_mask = series.isin(special_values)
    normal_mask = ~special_mask
    normal_series = series[normal_mask]

    final_bins = pd.Series(index=series.index, dtype="object")

    unique_vals = normal_series.nunique()

    # ---------------------------------
    # CASE 1: Low-cardinality numeric
    # ---------------------------------
    if unique_vals <= 6 and pd.api.types.is_numeric_dtype(normal_series):

        binned = normal_series.astype(str)
        bin_edges = sorted(normal_series.unique().tolist())

    # ---------------------------------
    # CASE 2: Continuous variable
    # ---------------------------------
    else:

        data_per_bin = normal_series.shape[0] * min_bin_size
        max_possible_bins = int(np.floor(normal_series.shape[0] / data_per_bin))
        n_bins = min(max_bins, max_possible_bins)

        try:
            binned, bin_edges = pd.qcut(
                normal_series,
                q=n_bins,
                retbins=True,
                duplicates="drop"
            )
        except ValueError:
            binned, bin_edges = pd.cut(
                normal_series,
                bins=min(5, normal_series.nunique()),
                retbins=True
            )

    final_bins.loc[normal_mask] = binned.astype(str)

    # Handle special codes
    for val in special_values:
        final_bins.loc[series == val] = f"SPECIAL_{val}"

    # ---------------------------------
    # Create summary table
    # ---------------------------------
    bin_table = (
        pd.DataFrame({
            "feature": feature,
            "Bin": final_bins,
            "target": y
        })
        .groupby(["feature", "Bin"], dropna=False)
        .agg(
            Count=("target", "count"),
            Event=("target", "sum")
        )
        .reset_index()
    )

    bin_table["Non-event"] = bin_table["Count"] - bin_table["Event"]

    total_events = bin_table["Event"].sum()
    total_non_events = bin_table["Non-event"].sum()

    bin_table["Count (%)"] = bin_table["Count"] / len(df)
    bin_table["Event rate"] = bin_table["Event"] / bin_table["Count"]

    bin_table["dist_event"] = bin_table["Event"] / total_events
    bin_table["dist_non_event"] = bin_table["Non-event"] / total_non_events

    # WoE
    bin_table["WoE"] = np.where(
        (bin_table["Event"] == 0) | (bin_table["Non-event"] == 0),
        0,
        np.log(bin_table["dist_non_event"] / bin_table["dist_event"])
    )

    # IV
    bin_table["IV"] = (
        (bin_table["dist_non_event"] - bin_table["dist_event"])
        * bin_table["WoE"]
    )

    total_iv = bin_table["IV"].sum()

    bin_table = bin_table.drop(columns=["dist_event", "dist_non_event"])

    # ---------------------------------
    # Sort bins
    # ---------------------------------

    normal_bins = bin_table[~bin_table["Bin"].astype(str).str.startswith("SPECIAL")]
    special_bins = bin_table[bin_table["Bin"].astype(str).str.startswith("SPECIAL")]

    if normal_bins["Bin"].astype(str).str.contains(",").any():

        normal_bins = normal_bins.copy()
        normal_bins["lower"] = normal_bins["Bin"].apply(
            lambda x: float(str(x).split(",")[0].replace("(", "").replace("[",""))
        )

        normal_bins = normal_bins.sort_values("lower").drop(columns="lower")

    else:
        normal_bins = normal_bins.sort_values("Bin")

    bin_table = pd.concat([normal_bins, special_bins]).reset_index(drop=True)

    return final_bins, bin_table, list(bin_edges), total_iv

In [ ]:

merge_plan = [
    [0,1],
    [2],
    [3,4],
    [5]
] # entrance median

In [7]:
id_col = 'SK_ID_CURR'
num_cols = (
        X_train.select_dtypes(include=[np.number])
        .select_dtypes(exclude=["bool"])
        .columns
        .drop(id_col)
    )


#apply
iv_list = []
summary_dict = {}
edges_dict = {}
bin_assignment_dict = {}
for feature in num_cols:
    
    try:
        bins, summary, edges, total_iv = manual_prebinning(
            df=X_train,
            feature=feature,
            target=y_train,
            max_bins=20,
            min_bin_size=0.05,
            special_values=[-99999, -88888, -77777]
        )
        
        iv_list.append({
            "feature": feature,
            "IV": total_iv
        })
        
        summary_dict[feature] = summary
        edges_dict[feature] = edges
        
        # 🔥 store bin assignments
        bin_assignment_dict[feature] = bins
        
    except Exception as e:
        print(f"Skipping {feature} due to error: {e}")


In [8]:
#genrate the woe_df 
woe_data = {}

for feature in num_cols:
    
    if feature not in summary_dict:
        continue
    
    summary = summary_dict[feature]
    woe_map = dict(zip(summary["Bin"], summary["WoE"]))
    
    woe_data[feature] = (
        bin_assignment_dict[feature]
        .map(woe_map)
    )

# Create dataframe in one shot
X_train_woe = pd.DataFrame(woe_data, index=X_train.index)
X_train_woe = X_train_woe.astype('float32')

In [9]:
iv_df = pd.DataFrame(iv_list)

# sort by IV descending
iv_df = iv_df.sort_values("IV", ascending=False).reset_index(drop=True)

features = iv_df['feature'].tolist()

In [10]:
id_col = X_train['SK_ID_CURR']
id_col

0         310536
1         365516
2         242055
3         454894
4         448321
           ...  
246003    136325
246004    240509
246005    387513
246006    303331
246007    430259
Name: SK_ID_CURR, Length: 246008, dtype: int64

In [11]:
final_df = X_train_woe[features].copy()
final_df['SK_ID_CURR'] = X_train['SK_ID_CURR'].values
final_df

,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,ANNUITY_CREDIT_RATIO,YEARS_EMPLOYED,AMT_GOODS_PRICE,IP_WORST_DPD_720D,YEARS_AGE,CB_MAX_MONTHLY_UTIL_6M,CB_MAX_MONTHLY_UTIL_12M,...,FLAG_DOCUMENT_20,FLAG_DOCUMENT_7,FLAG_DOCUMENT_5,FLAG_DOCUMENT_4,FLAG_DOCUMENT_10,FLAG_MOBIL,FLAG_DOCUMENT_12,CB_UTIL_VOLATILITY_1M,CB_STD_PAYMENT_VOLATILITY_1M,SK_ID_CURR
0,0.595258,-0.404168,-0.115018,0.100148,-0.251998,-0.051268,0.070145,-0.248158,0.430272,0.254160,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,310536
1,-0.152297,0.060750,-0.059399,-0.736789,-0.310259,0.241667,0.206623,-0.019086,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,365516
2,0.509141,-0.005531,-0.059399,0.122484,-0.106810,0.091518,0.206623,0.157868,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,242055
3,0.015118,0.571492,-0.059399,-0.254265,-0.106810,-0.555872,0.206623,0.126092,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,454894
4,-0.152297,-0.659605,-0.059399,0.135550,-0.251998,-0.105109,0.206623,-0.430459,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,448321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246003,-0.152297,0.109190,-0.059399,0.094759,-0.404514,-0.555872,0.206623,-0.019086,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,136325
246004,0.509141,0.218512,-0.059399,-0.736789,0.293260,-0.105109,-0.893440,0.157868,-0.052707,0.254160,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,240509
246005,0.750628,-0.082846,-0.059399,1.174794,0.436878,0.031894,-0.316884,0.428326,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,387513
246006,1.041111,-0.082846,-0.059399,0.070660,0.436878,0.285476,-0.629699,0.428326,0.032424,0.032424,...,0.000109,-0.000039,-0.000175,-0.000097,-0.000018,-0.000004,-0.000004,0.0,0.0,303331


In [16]:
numerical_feature_bins['REGION_RATING_CLIENT']

,feature,Bin,Count,Event,Non-event,Count (%),Event rate,WoE,IV
0,REGION_RATING_CLIENT,1,25755,1249,24506,0.104692,0.048495,0.544093,0.024741
1,REGION_RATING_CLIENT,2,181631,14313,167318,0.738313,0.078803,0.026246,0.000503
2,REGION_RATING_CLIENT,3,38622,4298,34324,0.156995,0.111284,-0.354787,0.022933
Totals,REGION_RATING_CLIENT,None,246008,19860,226148,1.000000,0.080729,NaN,0.048177


In [ ]:
iv_df.rename(columns={'IV':'score'},inplace=True)

In [ ]:
iv_df[iv_df['feature']=='ANNUITY_CREDIT_RATIO']

,feature,score
3,ANNUITY_CREDIT_RATIO,0.145948


In [17]:
iv_df.rename(columns={'score':'iv'},inplace=True)

In [18]:
from src.components.module_05_feature_engineering import FeatureSelector
fe = FeatureSelector()
kept_features = fe.iv_filteration(final_df,iv_df)

2026-03-10 07:48:16,045 - module_05_feature_engineering.py - INFO - Kept Feature 175 features with IV > 0.02
2026-03-10 07:48:16,045 - module_05_feature_engineering.py - INFO - Kept Feature 175 features with IV > 0.02
2026-03-10 07:48:16,050 - module_05_feature_engineering.py - INFO - Removed 123 features with IV < 0.02
2026-03-10 07:48:16,050 - module_05_feature_engineering.py - INFO - Removed 123 features with IV < 0.02


In [19]:
final_df = final_df[kept_features].copy()

In [20]:
final_df['SK_ID_CURR'] = X_train['SK_ID_CURR'].values
final_df

,EXT_SOURCE_3,EXT_SOURCE_2,EXT_SOURCE_1,ANNUITY_CREDIT_RATIO,YEARS_EMPLOYED,AMT_GOODS_PRICE,IP_WORST_DPD_720D,YEARS_AGE,CB_MAX_MONTHLY_UTIL_6M,CB_MAX_MONTHLY_UTIL_12M,...,CB_MAX_RATIO_POS_TO_TOTAL_DRAW_1M,NONLIVINGAREA_MEDI,NONLIVINGAREA_MODE,BASEMENTAREA_AVG,BASEMENTAREA_MEDI,CB_RATIO_UNDERPAYMENT_24M,BASEMENTAREA_MODE,CB_RATIO_UNDERPAYMENT_18M,CB_RATIO_MAX_POS_SPEND_6M,SK_ID_CURR
0,0.595258,-0.404168,-0.115018,0.100148,-0.251998,-0.051268,0.070145,-0.248158,0.430272,0.254160,...,0.028235,0.134548,0.136585,0.361320,0.354624,-0.541156,0.365470,-0.508105,0.033826,310536
1,-0.152297,0.060750,-0.059399,-0.736789,-0.310259,0.241667,0.206623,-0.019086,0.032424,0.032424,...,0.028235,-0.121890,-0.121890,-0.105908,-0.105908,0.032424,-0.105908,0.032424,0.032424,365516
2,0.509141,-0.005531,-0.059399,0.122484,-0.106810,0.091518,0.206623,0.157868,0.032424,0.032424,...,0.028235,0.134548,0.136585,0.141134,0.150370,0.032424,0.221880,0.032424,0.032424,242055
3,0.015118,0.571492,-0.059399,-0.254265,-0.106810,-0.555872,0.206623,0.126092,0.032424,0.032424,...,0.028235,0.134548,0.136585,0.061084,0.053228,0.032424,0.108148,0.032424,0.032424,454894
4,-0.152297,-0.659605,-0.059399,0.135550,-0.251998,-0.105109,0.206623,-0.430459,0.032424,0.032424,...,0.028235,-0.121890,-0.121890,-0.105908,-0.105908,0.032424,-0.105908,0.032424,0.032424,448321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
246003,-0.152297,0.109190,-0.059399,0.094759,-0.404514,-0.555872,0.206623,-0.019086,0.032424,0.032424,...,0.028235,-0.121890,-0.121890,-0.105908,-0.105908,0.032424,-0.105908,0.032424,0.032424,136325
246004,0.509141,0.218512,-0.059399,-0.736789,0.293260,-0.105109,-0.893440,0.157868,-0.052707,0.254160,...,0.028235,0.134548,0.136585,0.141134,0.150370,0.057806,0.221880,0.056244,0.033826,240509
246005,0.750628,-0.082846,-0.059399,1.174794,0.436878,0.031894,-0.316884,0.428326,0.032424,0.032424,...,0.028235,0.270455,0.247106,0.417202,0.413018,0.032424,0.381784,0.032424,0.032424,387513
246006,1.041111,-0.082846,-0.059399,0.070660,0.436878,0.285476,-0.629699,0.428326,0.032424,0.032424,...,0.028235,-0.121890,-0.121890,-0.105908,-0.105908,0.032424,-0.105908,0.032424,0.032424,303331


In [21]:
#iv_df.rename(columns={'iv':'score'},inplace=True)
temp_df = iv_df[iv_df['feature'].isin(kept_features)].copy()

In [22]:
import gc
del X_train,y_train
gc.collect()

0

In [23]:
#temp_df.set_index('feature', inplace=True)   

In [24]:
from src.components.module_05_feature_engineering import FeatureSelector
fe = FeatureSelector()
rr = fe.correlation_filter(final_df,kept_features,temp_df)

2026-03-10 07:48:35,864 - module_05_feature_engineering.py - INFO - Total features before : 175
2026-03-10 07:48:35,864 - module_05_feature_engineering.py - INFO - Total features before : 175
2026-03-10 07:48:35,868 - module_05_feature_engineering.py - INFO - Total features after  : 42
2026-03-10 07:48:35,868 - module_05_feature_engineering.py - INFO - Total features after  : 42
2026-03-10 07:48:35,872 - module_05_feature_engineering.py - INFO - Dropped features      : 133
2026-03-10 07:48:35,872 - module_05_feature_engineering.py - INFO - Dropped features      : 133
2026-03-10 07:48:35,885 - module_05_feature_engineering.py - INFO - Removed 133 features due to high correlation (> 0.7)
2026-03-10 07:48:35,885 - module_05_feature_engineering.py - INFO - Removed 133 features due to high correlation (> 0.7)
2026-03-10 07:48:35,889 - module_05_feature_engineering.py - INFO - Remaining feature list final shape is  42
2026-03-10 07:48:35,889 - module_05_feature_engineering.py - INFO - Remain

In [25]:
iv_df[iv_df['feature'].isin(rr)]

,feature,iv
0,EXT_SOURCE_3,0.336897
1,EXT_SOURCE_2,0.315593
2,EXT_SOURCE_1,0.156080
3,ANNUITY_CREDIT_RATIO,0.146553
4,YEARS_EMPLOYED,0.114435
5,AMT_GOODS_PRICE,0.102669
6,IP_WORST_DPD_720D,0.091014
7,YEARS_AGE,0.088899
8,CB_MAX_MONTHLY_UTIL_6M,0.088130
22,GOODS_CREDIT_RATIO,0.076527


In [26]:

#SElected Features
with open(r'D:\home loan credit risk\artifact\binning\automatic\selected_features.json', 'rb') as f:
    selected_features = json.load(f)
for i in range(len(selected_features)):
    selected_features[i] = selected_features[i].replace('_WOE','')
        
#X_train_woe = pd.read_csv(r'D:\home loan credit risk\artifact\binning\automatic\X_train_selected_woe.csv')


In [36]:
tp = pd.read_csv(r'D:\home loan credit risk\artifact\binning\automatic\iv_df.csv')

# old class optbinning woe binner numerical

In [ ]:

#-----------------------------------------------------------------
class NumericalWOEBinner:
    """
    Numerical WOE Binning using optbinning
    - Fit on TRAIN only
    - Transform TRAIN / TEST safely
    """
    def __init__(self, id_col, special_codes=None, max_n_bins=20, min_prebin_size=0.05,prebinning_method='quantile'):
        '''
        Initialize WOE binning configuration.
        params
            id_col: id col from dataset
            special_codes: special values in dict to treat them as bin
            max_n_bins: Maximum number of final bins per variable
            min_prebin_size: Minimum proportion of observations per pre-bin
            prebinning_method: Strategy used for initial bin creation (e.g., 'quantile')

        '''
        self.id_col = id_col
        self.special_codes = special_codes
        self.max_n_bins = max_n_bins
        self.min_prebin_size = min_prebin_size
        self.prebinning_method = prebinning_method
        self.binning_models = {}
        self.iv_df = None
        self.numerical_features = []

    def fit(self, X_train_num, y_train):
        """ Fit WOE binning models using TRAINING data only
        fits the optbinning model per numerical variable  calculate the bin ,woe, and iv for features
        stores binning model.

        Params:
            X_train_num: pd.DataFrame containing only numerical features including id_col
            y_train : Binary target variable (event / non-event)

        """
        self.numerical_features = [c for c in X_train_num.columns if c != self.id_col]
        iv_records = []

        for col in self.numerical_features:
            try:
                optb = OptimalBinning(
                    name=col,
                    dtype="numerical",
                    solver="cp",
                    prebinning_method=self.prebinning_method,
                    min_prebin_size=self.min_prebin_size,
                    max_n_bins=self.max_n_bins,
                    special_codes=self.special_codes
                )

                # Fit on TRAIN only
                optb.fit(X_train_num[col], y_train)

                # Build binning table
                bt = optb.binning_table
                bt.build()

                # Store model
                self.binning_models[col] = optb

                # Store IV
                iv_records.append({
                    "feature": col,
                    "iv": bt.iv
                })

            except MyException as e:
                logger.warning(f"Skipped {col}: {e}")

        self.iv_df = pd.DataFrame(iv_records).sort_values(by="iv", ascending=False).reset_index(drop=True)
        return self

    def transform(self, X_num):
        '''
        Apply pre-fitted WOE binning rules to any dataset.
        - Missing values are mapped to a fixed placeholder to ensure


        params:
            X_num:  Dataset to be transformed (Train / Validation / Test)
            
        Returns:
            X_woe: WOE-transformed numerical features with consistent binning
        '''
        X_woe = pd.DataFrame(index=X_num.index)
        X_woe[self.id_col] = X_num[self.id_col]

        for col, optb in self.binning_models.items():
            if col not in X_num.columns:
                continue                                    # placeholder for missing value
            X_woe[col + "_WOE"] = optb.transform(X_num[col].fillna(-99999), metric="woe")

        return X_woe

    def fit_transform(self, X_train_num, y_train):
        '''Fit WOE bins using training data and
        return WOE-transformed training features. call strictly on for the training data
        
        params:
            X_train_num: pd.DataFrame containing only numerical features including id_col
            y_train : Binary target variable (event / non-event)
            
        Returns:
            X_woe: WOE-transformed numerical features with consistent binning
        '''
        self.fit(X_train_num, y_train)
        return self.transform(X_train_num)
    
    def get_numerical_bins_dict(self):
        """
        Generate a dictionary of binning DataFrames for each numerical feature.

        Args:
            num_binner (NumericalWOEBinner): Fitted numerical WOE binner

        Returns:
            dict: {feature_name: binning_table_df}
        """
        self.feature_bins_num = {}

        for feature, optb in self.binning_models.items():
            bt = optb.binning_table
            bt_df = bt.build()
            bt_df["feature"] = feature
            self.feature_bins_num[feature] = bt_df
            
        return self.feature_bins_num
    

In [31]:
# CODE IS FOR THE MANUAL BINNING WOE _DF DIRECT UPDATE PREVIOUS VERSION
'''import numpy as np

def update_merge_bins(feature,bins_index:list, woe_df, feature_bins_num):
this function direcly merges the bins and update the values in the woe_df and also rows_df
    
    bins_index = sorted(bins_index)

    feature_df = feature_bins_num[feature]
    bin_filter = feature_df.index.isin(bins_index)

    # filter count bins
    count_bin = feature_df.loc[bin_filter,'Count'].sum()
    count_non_event = feature_df.loc[bin_filter,'Non-event'].sum()
    count_event = feature_df.loc[bin_filter,'Event'].sum()
    # filter bin rate
    bin_event_rate = count_event / count_bin

    
    # total df event and non event
    total_df_event = feature_df['Event'].sum()
    total_df_non_event = feature_df['Non-event'].sum()

    # WOE Calculation
    dist_non_event = np.float64(count_non_event) / np.float64(total_df_non_event)
    dist_event = np.float64(count_event) / np.float64(total_df_event)

    # WOE
    merged_woe = np.log(dist_non_event / dist_event)
    
    # iv 
    merged_iv = (dist_non_event - dist_event) * merged_woe
    
    #old_iv_sum = feature_df.loc[bin_filter, 'IV'].sum()
    #total_feature_iv = feature_df['IV'].sum() - old_iv_sum + merged_iv

    feature_woe = feature+ '_WOE'
    old_woe_values = (
        pd.to_numeric(feature_df.loc[bin_filter, 'WoE'], errors='coerce')
        .round(6)
        .dropna()
        .tolist()
    )

    filt = (
        pd.to_numeric(woe_df[feature_woe], errors='coerce')
        .round(6)
        .isin(old_woe_values)
    )

    # updating woe in woe_Df
    woe_df.loc[filt,feature_woe] = merged_woe

    # update the feature num bins
    temp_num_bin = feature_bins_num[feature]
    lower = temp_num_bin.loc[bins_index[0], 'Bin'].split(',')[0].replace('(', '')  
    upper = temp_num_bin.loc[bins_index[-1], 'Bin'].split(',')[1].replace(')', '')
    upper = upper.strip()
    bin = f'({lower}, {upper}]'
    
  
    temp_num_bin.drop(index=bins_index[1:], inplace=True)
    total_count_updated = temp_num_bin.loc[
    temp_num_bin.index != 'Totals', 'Count'
    ].sum()
    count_pct = count_bin /total_count_updated

    
    updates = {
            'Bin': bin,
            'Count': count_bin,
            'Count (%)':count_pct,
            'Non-event':count_non_event,
            'Event':count_event,
            'Event rate':bin_event_rate,
            'WoE':merged_woe,
            'IV':merged_iv,
            'feature':feature
        }
    keep_idx = bins_index[0]
    #temp_num_bin.loc['Totals', 'IV'] = total_feature_iv
    
   
    temp_num_bin.loc[keep_idx, list(updates.keys())] = list(updates.values())


    return woe_df

def recompute_total_iv(feature, feature_bins_num):
    df = feature_bins_num[feature]
    total_iv = df.loc[df.index != 'Totals', 'IV'].sum()
    df.loc['Totals', 'IV'] = total_iv
    
def run_merge_plan(
    feature: str,
    merge_plan: list,
    woe_df,
    feature_bins_num,
    
    ):

    merge_plan example:
    [
        [0,1,2],
        [3,4],
        [5,6],
        [7,8,9],
        [10,11,12],
        [13,14,15,16]
    ]


    for bins_index in merge_plan:
        woe_df = update_merge_bins(
            feature=feature,
            bins_index=bins_index,
            woe_df=woe_df,
            feature_bins_num=feature_bins_num,
        )

    recompute_total_iv(feature, feature_bins_num)
    return woe_df
    '''

"import numpy as np\n\ndef update_merge_bins(feature,bins_index:list, woe_df, feature_bins_num):\nthis function direcly merges the bins and update the values in the woe_df and also rows_df\n\n    bins_index = sorted(bins_index)\n\n    feature_df = feature_bins_num[feature]\n    bin_filter = feature_df.index.isin(bins_index)\n\n    # filter count bins\n    count_bin = feature_df.loc[bin_filter,'Count'].sum()\n    count_non_event = feature_df.loc[bin_filter,'Non-event'].sum()\n    count_event = feature_df.loc[bin_filter,'Event'].sum()\n    # filter bin rate\n    bin_event_rate = count_event / count_bin\n\n\n    # total df event and non event\n    total_df_event = feature_df['Event'].sum()\n    total_df_non_event = feature_df['Non-event'].sum()\n\n    # WOE Calculation\n    dist_non_event = np.float64(count_non_event) / np.float64(total_df_non_event)\n    dist_event = np.float64(count_event) / np.float64(total_df_event)\n\n    # WOE\n    merged_woe = np.log(dist_non_event / dist_event)\n\